In [ ]:
# Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import CategoricalDtype
from pathlib import Path
from functools import reduce

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Update data_path to reflect the mounted Google Drive path
data_path = "/content/drive/My Drive/STAT390 Data/All Calls by Month/"

In [ ]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

### Reading first 5 rows of all data files
The code chunk below reads the first 5 rows of all data files. This is to check the columns that are present in all the data files.

In [ ]:
i=0; df = []
for f in files:
    if f.suffix.lower() == ".csv":
        df.append(pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False, nrows = 5))
    else:  # .xlsx
        df.append(pd.read_excel(f, sheet_name=0, header=0, dtype=str, nrows = 5))
    #df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df[i].shape)
    i = i + 1

0 April 2024 (5, 63)
1 April 2025 (5, 63)
2 August 2024 (5, 57)
3 August 2025 (5, 63)
4 December 2024 (5, 55)
5 February 2025 (5, 63)
6 January 2025 (5, 64)
7 July 2024 (5, 55)
8 July 2025 (5, 63)
9 June 2024 (5, 63)
10 June 2025 (5, 63)
11 March 2025 (5, 64)
12 May 2024 (5, 63)
13 May 2025 (5, 63)
14 November 2024 (5, 63)
15 October 2024 (5, 55)
16 September 2024 (5, 55)
17 September 2025 (5, 69)


The code chunk below identifies the columns missing in at least one DataFrame.

In [ ]:
all_cols = reduce(lambda x, y: x | set(y.columns), df[1:], set(df[0].columns))
common_cols = reduce(lambda x, y: x & set(y.columns), df[1:], set(df[0].columns))
common_cols
not_in_all = all_cols - common_cols
print("Columns missing from at least one dataframe:", not_in_all)

Columns missing from at least one dataframe: {'Public Called IP Address', 'Call Recording Result', 'Call Recording Platform Name', 'Recall Type', 'External caller ID number', 'Auto Attendant Key Pressed', 'Column1', 'Answered Elsewhere', 'Redirecting party UUID', 'PSTN vendor name2', 'Device owner UUID', 'User', 'Hold Duration', 'Call Recording Trigger', 'Original called party UUID', 'Queue Type', 'Public Calling IP Address'}


The code chunk below prints the columns present in all the data files.

In [ ]:
print(common_cols)

{'Answer Indicator', 'Releasing party', 'Release time', 'Transfer related call ID', 'International Country', 'Model', 'User number', 'Related call ID', 'Local SessionID', 'Related reason', 'PSTN provider ID', 'Duration', 'Answer time', 'Final local sessionID', 'Local call ID', 'Called number', 'Direction', 'Remote call ID', 'Org UUID', 'Device Mac', 'Site main number', 'Call ID', 'Redirect reason', 'Report ID', 'Outbound trunk', 'Location', 'Ring duration', 'Remote SessionID', 'PSTN legal entity', 'OS type', 'Call outcome reason', 'Report time', 'Inbound trunk', 'Start time', 'Call transfer time', 'PSTN vendor name', 'Client version', 'User UUID', 'Authorization code', 'Site UUID', 'Sub client type', 'Client type', 'Department ID', 'Original reason', 'Site timezone', 'Call outcome', 'Final remote sessionID', 'Route group', 'Answered', 'Redirecting number', 'Correlation ID', 'Call type', 'PSTN vendor Org ID', 'Network call ID', 'User type'}


### Reading all the data files
All the datafiles are read with the common columns read first.

In [ ]:
df_main = pd.DataFrame(columns=list(common_cols))

In [ ]:
i=0;
for f in files:
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=0, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)
    i = i + 1

0 April 2024 (56662, 63)
1 April 2025 (63636, 63)
2 August 2024 (63262, 57)
3 August 2025 (57071, 63)
4 December 2024 (49445, 55)
5 February 2025 (63669, 63)
6 January 2025 (62623, 64)
7 July 2024 (62292, 55)
8 July 2025 (60438, 63)
9 June 2024 (56763, 63)
10 June 2025 (54598, 63)
11 March 2025 (59149, 64)
12 May 2024 (62944, 63)
13 May 2025 (55428, 63)
14 November 2024 (49953, 63)
15 October 2024 (62354, 55)
16 September 2024 (61250, 55)
17 September 2025 (56571, 69)


### Converting date to datetime format

In [ ]:
df_main["Start time"] = pd.to_datetime(df_main["Start time"], utc=True)

In [ ]:
df_main["Start time"] = df_main["Start time"].dt.tz_convert("America/Chicago").dt.tz_localize(None)

In [ ]:
df_main["Start time"].head()

,Start time
0,2024-04-30 18:58:53.988
1,2024-04-30 18:56:37.386
2,2024-04-30 18:54:59.099
3,2024-04-30 18:54:59.099
4,2024-04-30 18:54:52.336


In [ ]:
df_allcallsdata = df_main.copy()

# Updated Steps

## Step 1: Group by "Correlation ID" and "Start time"

In [ ]:
df_allcallsdata.sort_values(by=['Correlation ID', 'Start time'], ascending=[True, True])[['Correlation ID', 'Start time', 'Called number', 'Duration']].head(50)

,Correlation ID,Start time,Called number,Duration
600224,00000fe9-dfa0-41c5-986b-d9ce731f2715,2025-06-27 11:25:23.791,13123411070,4
929258,00001e73-ce33-48f1-b531-bddc7ee3965d,2024-10-05 12:30:09.903,13124312299,1961
267542,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 14:41:47.355,13123478311,44
267540,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 14:42:05.358,13123478300,44
267541,00006ccc-8250-4993-90c0-bbfd41f7dd24,2024-12-11 14:42:05.358,13123478300,44
818656,00008bec-c404-48cc-8276-1f276bf4229a,2025-05-06 13:04:29.746,13122296344,65
818654,00008bec-c404-48cc-8276-1f276bf4229a,2025-05-06 13:04:29.749,13123478300,65
818655,00008bec-c404-48cc-8276-1f276bf4229a,2025-05-06 13:04:29.749,13123478300,65
974134,000090ae-a71c-49e9-99d6-fdc7078dfa48,2024-09-13 14:35:32.428,17086568223,57
828444,0000aef0-6f35-4560-a4be-5b3bc31fbefb,2024-11-28 09:49:47.564,13123411070,2


## Step 2: Extract the inbound calls

In [ ]:
### Remove call duration = 0
# convert the "Duration" column into numeric
df_allcallsdata["Duration"] = pd.to_numeric(df_allcallsdata["Duration"], errors="coerce")
df_allcallsdata = df_allcallsdata[df_allcallsdata["Duration"] > 0]


### Sort by Correlation ID and Start Time (chronological order)
df_allcallsdata = df_allcallsdata.sort_values(by=["Correlation ID", "Start time"])


### Classify call type (Inbound/Outbound/Internal)
def classify_call(row):
    if row["PSTN vendor name"] == "CallTower" and row["Direction"] == "ORIGINATING":
        return "Outbound"
    elif row["PSTN vendor name"] == "CallTower" and row["Direction"] == "TERMINATING":
        return "Inbound"
    elif row["PSTN vendor name"] == "NA":
        return "Internal"
    else:
        return "Other"

# Temporary column for per-row classification
df_allcallsdata["TempCallType"] = df_allcallsdata.apply(classify_call, axis=1)

# Propagate earliest-leg call type to all legs of the same call
earliest_calltype = df_allcallsdata.groupby("Correlation ID").first().reset_index()[["Correlation ID", "TempCallType"]]
df_allcallsdata = df_allcallsdata.drop(columns=["TempCallType"])
df_allcallsdata = df_allcallsdata.merge(earliest_calltype.rename(columns={"TempCallType":"Inbound/Outbound"}),
                                        on="Correlation ID", how="left")

In [ ]:
# Keep only the inbound calls
df_allcallsdata_inbound = df_allcallsdata[df_allcallsdata["Inbound/Outbound"] == "Inbound"]

In [ ]:
# Get a general sense of the information kept
print('Number of observations in the original dataset:', df_allcallsdata.shape[0])
print('Number of observations in the inbound call dataset:', df_allcallsdata_inbound.shape[0])
print('Proportion of observations kept:', df_allcallsdata_inbound.shape[0] / df_allcallsdata.shape[0])

Number of observations in the original dataset: 1003089
Number of observations in the inbound call dataset: 757494
Proportion of observations kept: 0.755161306723531


In [ ]:
# Extract time features
df_allcallsdata_inbound["Start time"] = pd.to_datetime(df_allcallsdata_inbound["Start time"])
df_allcallsdata_inbound["Hour"] = df_allcallsdata_inbound["Start time"].dt.hour
df_allcallsdata_inbound["DayOfWeek"] = df_allcallsdata_inbound["Start time"].dt.weekday + 1  # Monday=1, Sunday=7
df_allcallsdata_inbound["Month"] = df_allcallsdata_inbound["Start time"].dt.month
df_allcallsdata_inbound["Quarter"] = df_allcallsdata_inbound["Start time"].dt.quarter
df_allcallsdata_inbound["Year"] = df_allcallsdata_inbound["Start time"].dt.year

/tmp/ipython-input-2771028058.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allcallsdata_inbound["Start time"] = pd.to_datetime(df_allcallsdata_inbound["Start time"])
/tmp/ipython-input-2771028058.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allcallsdata_inbound["Hour"] = df_allcallsdata_inbound["Start time"].dt.hour
/tmp/ipython-input-2771028058.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = valu

## Step 3: Add a column to describes the numbers' functions

In [ ]:
number_map = {
    # From "QuestionsForCynthia_30Sep_Krish"
    "13123478300": "Internal voicemail - not client related",
    "13123411070": "Main number",
    "13124235938": "Community Legal Clinics",
    "13122296300": "Direct Line to Front Desk",
    "1180": "Transfers to English Queue Options (Legal Menu)",
    "13125068646": "Transfers to the English main menu",
    "13124312299": "Farmworker main number / Migrant Legal Assistance Program",
    "13122296079": "Nursing Home Ombudsman",
    "13122296344": "Bankruptcy Helpdesk Voicemail",
    "13122296071": "Criminal Records",
    "13125068647": "Transfers to the Spanish main menu",
    "13123478340": "Veterans Rights Project Voicemail",
    "13122296014": "Markham Eviction Help Desk",
    "13123478309": "HIV Intake Voicemail",
    "13122296072": "Juvenile Expungement Help Desk (JEHD)",
    "13124235904": "Austin Intake Voicemail",

    # From "Intake phone numbers for reporting"
    "13123478347": "A2J Immigration (Lisa Palumbo's direct line)",
    "18882652188": "A2J Immigration (Lisa Palumbo's direct line)",  # rings to 13123478347
    "13124235900": "CLASP Voicemail",
    "13123478392": "Education Law Referrals Voicemail",
    "13124235909": "Fair Housing Intake Voicemail",
    "13124312101": "OP Appeals Project",
    "13122296073": "Trafficking Survivors Assistance Project (TSAP)",
    "18004459025": "Migrant Legal Assistance Program",  # rings to 13124312299
    "18884018200": "Nursing Home Ombudsman"  # rings to 13122296079
}


In [ ]:
# Add the number description column for better visualization
df_allcallsdata_inbound["Number Description"] = df_allcallsdata_inbound["Called number"].astype(str).map(number_map)

# Replace unmapped numbers with a default label
df_allcallsdata_inbound["Number Description"] = df_allcallsdata_inbound["Number Description"].fillna("Unknown")

/tmp/ipython-input-912251125.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allcallsdata_inbound["Number Description"] = df_allcallsdata_inbound["Called number"].astype(str).map(number_map)
/tmp/ipython-input-912251125.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_allcallsdata_inbound["Number Description"] = df_allcallsdata_inbound["Number Description"].fillna("Unknown")


In [ ]:
df_allcallsdata_inbound["Number Description"].unique()

array(['Main number',
       'Farmworker main number / Migrant Legal Assistance Program',
       'Unknown', 'Internal voicemail - not client related',
       'Bankruptcy Helpdesk Voicemail', 'Community Legal Clinics',
       'Austin Intake Voicemail', 'Criminal Records',
       'Direct Line to Front Desk', 'Nursing Home Ombudsman',
       'Veterans Rights Project Voicemail',
       'Transfers to English Queue Options (Legal Menu)',
       'Transfers to the English main menu', 'Markham Eviction Help Desk',
       'Transfers to the Spanish main menu', 'HIV Intake Voicemail',
       'Juvenile Expungement Help Desk (JEHD)',
       'Fair Housing Intake Voicemail', 'OP Appeals Project',
       'CLASP Voicemail', "A2J Immigration (Lisa Palumbo's direct line)",
       'Trafficking Survivors Assistance Project (TSAP)',
       'Education Law Referrals Voicemail'], dtype=object)

In [ ]:
df_allcallsdata_inbound["Number Description"].value_counts()

,count
Number Description,
Internal voicemail - not client related,222215
Main number,211793
Unknown,173312
Community Legal Clinics,52411
Direct Line to Front Desk,34026
Transfers to English Queue Options (Legal Menu),10725
Transfers to the English main menu,10324
Farmworker main number / Migrant Legal Assistance Program,9728
Bankruptcy Helpdesk Voicemail,6882


## Step 4: Map out the call flow for each call (for drill-down feature)

In [ ]:
# Ensure proper sorting by correlation and time
df_allcallsdata_inbound = df_allcallsdata_inbound.sort_values(["Correlation ID", "Start time"])

# Group by call and collect all descriptions (keep 'Unknown')
call_sequences = (
    df_allcallsdata_inbound.groupby("Correlation ID")["Number Description"]
    .apply(lambda x: [v for v in x if pd.notna(v)])
    .reset_index(name="Call Flow")
)

# Add num_legs column
call_sequences["num_legs"] = call_sequences["Call Flow"].apply(len)

# Remove calls with unusually high number of legs (outliers)
call_sequences_clean = call_sequences[call_sequences["num_legs"] <= 4].copy()

# Find the max number of legs to know how many columns to create
max_legs = call_sequences_clean["Call Flow"].apply(len).max()

# Expand the sequence horizontally
for i in range(max_legs):
    call_sequences_clean[f"Number Description_{i+1}"] = call_sequences_clean["Call Flow"].apply(
        lambda x: x[i] if len(x) > i else None
    )

# Preview result
call_sequences_clean.head(10)

,Correlation ID,Call Flow,num_legs,Number Description_1,Number Description_2,Number Description_3,Number Description_4
0,00000fe9-dfa0-41c5-986b-d9ce731f2715,[Main number],1,Main number,None,None,None
1,00001e73-ce33-48f1-b531-bddc7ee3965d,[Farmworker main number / Migrant Legal Assist...,1,Farmworker main number / Migrant Legal Assista...,None,None,None
2,00006ccc-8250-4993-90c0-bbfd41f7dd24,"[Unknown, Internal voicemail - not client rela...",3,Unknown,Internal voicemail - not client related,Internal voicemail - not client related,None
3,00008bec-c404-48cc-8276-1f276bf4229a,"[Bankruptcy Helpdesk Voicemail, Internal voice...",3,Bankruptcy Helpdesk Voicemail,Internal voicemail - not client related,Internal voicemail - not client related,None
4,0000aef0-6f35-4560-a4be-5b3bc31fbefb,[Main number],1,Main number,None,None,None
5,0000ff52-cebd-430b-bca8-b8ae9a8b6e04,[Main number],1,Main number,None,None,None
6,0001424d-5229-46a5-9661-04dbc11e5273,"[Unknown, Internal voicemail - not client rela...",3,Unknown,Internal voicemail - not client related,Internal voicemail - not client related,None
8,000197f8-fafe-46d3-a830-94aefc90fd90,[Internal voicemail - not client related],1,Internal voicemail - not client related,None,None,None
9,0001af3f-5951-4d02-bf61-27cab9793b9d,"[Unknown, Internal voicemail - not client rela...",3,Unknown,Internal voicemail - not client related,Internal voicemail - not client related,None
10,0001bba9-cd7c-4e4d-94be-c54e880c10c5,[Main number],1,Main number,None,None,None


In [ ]:
# Examine the proportion for each called number in the first step
call_sequences_clean['Number Description_1'].value_counts()

,count
Number Description_1,
Main number,161016
Unknown,67020
Farmworker main number / Migrant Legal Assistance Program,9477
Bankruptcy Helpdesk Voicemail,6595
Nursing Home Ombudsman,6392
Community Legal Clinics,3568
Internal voicemail - not client related,3138
Veterans Rights Project Voicemail,2211
Criminal Records,2024


In [ ]:
# Examine the proportion for each called number during the first transfe
call_sequences_clean['Number Description_2'].value_counts()

,count
Number Description_2,
Internal voicemail - not client related,69013
Unknown,22225
Direct Line to Front Desk,5040
Main number,1470
Farmworker main number / Migrant Legal Assistance Program,102
Markham Eviction Help Desk,45
Bankruptcy Helpdesk Voicemail,37
Community Legal Clinics,14
Transfers to the English main menu,7


In [ ]:
call_sequences_clean['Number Description_3'].value_counts()

,count
Number Description_3,
Internal voicemail - not client related,68976
Unknown,21547
Direct Line to Front Desk,5028
Markham Eviction Help Desk,42
Community Legal Clinics,6
Nursing Home Ombudsman,1
Transfers to English Queue Options (Legal Menu),1


In [ ]:
call_sequences_clean['Number Description_4'].value_counts()

,count
Number Description_4,
Internal voicemail - not client related,10
Unknown,3
Bankruptcy Helpdesk Voicemail,2
Transfers to the English main menu,1


In [ ]:
inbound_call_workflow = call_sequences_clean.copy()

## Step 5: Merge with the meaningful columns from the original dataset

In [ ]:
df_allcallsdata_inbound.columns

Index(['Answer Indicator', 'Releasing party', 'Release time',
       'Transfer related call ID', 'International Country', 'Model',
       'User number', 'Related call ID', 'Local SessionID', 'Related reason',
       'PSTN provider ID', 'Duration', 'Answer time', 'Final local sessionID',
       'Local call ID', 'Called number', 'Direction', 'Remote call ID',
       'Org UUID', 'Device Mac', 'Site main number', 'Call ID',
       'Redirect reason', 'Report ID', 'Outbound trunk', 'Location',
       'Ring duration', 'Remote SessionID', 'PSTN legal entity', 'OS type',
       'Call outcome reason', 'Report time', 'Inbound trunk', 'Start time',
       'Call transfer time', 'PSTN vendor name', 'Client version', 'User UUID',
       'Authorization code', 'Site UUID', 'Sub client type', 'Client type',
       'Department ID', 'Original reason', 'Site timezone', 'Call outcome',
       'Final remote sessionID', 'Route group', 'Answered',
       'Redirecting number', 'Correlation ID', 'Call type',
   

In [ ]:
# Extract first leg info for each call
first_leg_info = df_allcallsdata_inbound.groupby("Correlation ID").first().reset_index()

# Select only the columns I want to merge
first_leg_info = first_leg_info[["Correlation ID", "Hour", "DayOfWeek", "Month", "Quarter", "Year", "Start time"]]

# Step 4: Merge with call_sequences_clean
call_sequences_merged = call_sequences_clean.merge(first_leg_info, on="Correlation ID", how="left")

# Preview
call_sequences_merged.head()


,Correlation ID,Call Flow,num_legs,Number Description_1,Number Description_2,Number Description_3,Number Description_4,Hour,DayOfWeek,Month,Quarter,Year,Start time
0,00000fe9-dfa0-41c5-986b-d9ce731f2715,[Main number],1,Main number,None,None,None,11,5,6,2,2025,2025-06-27 11:25:23.791
1,00001e73-ce33-48f1-b531-bddc7ee3965d,[Farmworker main number / Migrant Legal Assist...,1,Farmworker main number / Migrant Legal Assista...,None,None,None,12,6,10,4,2024,2024-10-05 12:30:09.903
2,00006ccc-8250-4993-90c0-bbfd41f7dd24,"[Unknown, Internal voicemail - not client rela...",3,Unknown,Internal voicemail - not client related,Internal voicemail - not client related,None,14,3,12,4,2024,2024-12-11 14:41:47.355
3,00008bec-c404-48cc-8276-1f276bf4229a,"[Bankruptcy Helpdesk Voicemail, Internal voice...",3,Bankruptcy Helpdesk Voicemail,Internal voicemail - not client related,Internal voicemail - not client related,None,13,2,5,2,2025,2025-05-06 13:04:29.746
4,0000aef0-6f35-4560-a4be-5b3bc31fbefb,[Main number],1,Main number,None,None,None,9,4,11,4,2024,2024-11-28 09:49:47.564


# Export the dataset

In [ ]:
inbound_call_workflow = call_sequences_merged

In [ ]:
inbound_call_workflow.to_csv("Nov4_AllCallsData_Inbound Call Workflow.csv", index=False)